- Welcome to Quantitative and High Frequency Trading Training Program
- In this week, we will


- Add rolling maximum and minimum to the data
- Test 5 predictive signals and generate each result
- Compare the resutls to see which signal perform the best
- Compare each day's result to check which days perform the best or worst
- Make some adjustments to the signals and check out the result
- Repeat the process to another data set I give you, each person may have different data sets


- We can repeat some of the process of week 2


- Change the path of helper.r if needed

In [ ]:
from helper import *
from stats import *
from product_info import *
from imp import reload  
import helper
import stats
reload(helper)
reload(stats)

 number of cores, set this number according to your computer

In [ ]:
CORE_NUM = int(os.environ['NUMBER_OF_PROCESSORS'])
CORE_NUM

In [ ]:
import warnings
 
warnings.filterwarnings('ignore')

- sample product, you can change to your specific product symbol

In [ ]:
product_list = ["rb", "hc", "i", "j", "jm"]

- get all of the dates

In [ ]:
import dask
from dask import compute, delayed

- get good night date

- now it's see why we use this kind of data

In [ ]:
all_dates = os.listdir(DATA_PATH + product_list[0])
all_dates.sort()
n_days = len(all_dates)


- we randomly choose a day

In [ ]:
data = load(DATA_PATH+product_list[0]+"/"+all_dates[0])

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(1, figsize=(16, 10))
plt.plot(data["wpr"].values);

- plot the maximum price of 2048 ticks
- plot the minimum price of 2048 ticks

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(1, figsize=(16, 10))
plt.plot(data["wpr"].values)
plt.plot(data["max.4096"].values)
plt.plot(data["min.4096"].values);

## we can see the result meets our expectation
## it may take long time to calculate the min and max of wpr prices
## I give you the estimated time of using 4 cores, so you can deal with other affairs while waiting for result
## but signals using min/max would be sometimes very useful
## the the time to calculating min/max pay off in the end
## In practice, when using c++ to calculating min/max on a rolling basis
## there would be some efficient algorithm, which is faster than comparing all numbers at every tick
## we would talk about it later in our C++ class

- Test 5 predictive signals and generate each result
- now let's see the first signal, which is simply related to price action without trading volume


- set the sample period length

In [ ]:
period = 4096

- the first signal

In [ ]:
from collections import OrderedDict
class foctor_dbook_period(factor_template):
    factor_name = "dbook.period"
    
    params = OrderedDict([
        ("period", np.power(2, range(12,13)))
    ])
    
    def formula(self, data, period):
        diff_bid_qty = data["bid.qty"] - data["bid.qty"].shift()
        diff_bid_qty[0] = 0
        diff_ask_qty = data["ask.qty"] - data["ask.qty"].shift()
        diff_ask_qty[0] = 0
        return ewma(np.sign(diff_bid_qty)-np.sign(diff_ask_qty), period, adjust=True).values

In [ ]:
x3 = foctor_dbook_period()
x3

# factor_name:
# dbook.period

# formula:
#     def formula(self, data, period):
#         diff_bid_qty = data["bid.qty"] - data["bid.qty"].shift()
#         diff_bid_qty[0] = 0
#         diff_ask_qty = data["ask.qty"] - data["ask.qty"].shift()
#         diff_ask_qty[0] = 0
#         return ewma(np.sign(diff_bid_qty)-np.sign(diff_ask_qty), period, adjust=True).values


# params:
# $period:[4096]

- create directory for dbook signal

In [ ]:
SAVE_PATH = "e:/intern"

In [ ]:
for product in product_list:
    create_signal_path(x3, product, SAVE_PATH)

In [ ]:
%%time
for product in product_list:
    file_list = list(map(lambda x: DATA_PATH+product+"/"+x, os.listdir(DATA_PATH + product)))
    parLapply(CORE_NUM, file_list, build_composite_signal,signal_list=x3, product=product, HEAD_PATH=SAVE_PATH)
;

#Wall time: 10min 44s

- now we write the all.signal part into a function

In [ ]:
def get_all_signal(file_list, product, signal_name, period):
    n_files = len(file_list)
    to_choose = (np.arange(n_files)+1) % 10 == 0
    all_signal = np.array([])
    for file in file_list[to_choose]:
        S = load(SAVE_PATH+"/tmp pkl/"+product+"/"+signal_name+"/"+file)
        data = load(HEAD_PATH+"/pkl tick/"+product+"/"+file)
        signal = S[data["good"]]
        chosen = (np.arange(len(signal))+1) % period==0
        all_signal = np.concatenate((all_signal, signal[chosen]), axis=0)
    return all_signal

In [ ]:
i = 0
for product in product_list:
    all_signal = get_all_signal(np.array(all_dates), product, "dbook."+str(period), period)
    i = i+1
    plt.figure(i, figsize=(16, 10))
    plt.title(product)
    plt.plot(all_signal)

In [ ]:
print(len(all_signal))
## 609


- there is no extreme values

In [ ]:
from collections import OrderedDict
import itertools
open_list = np.arange(0.05, 0.15, 0.005)
thre_list = []
for cartesian in itertools.product(open_list, np.array([0.2, 0.4, 0.6, 0.8, 1.0])):
    thre_list.append((cartesian[0], -cartesian[0] * cartesian[1]))
thre_list = np.array(thre_list)
thre_mat = pd.DataFrame(data=OrderedDict([("open", thre_list[:, 0]), ("close", thre_list[:, 1])]))
np.shape(thre_mat)

- get the result of training sample

In [ ]:
period = 4096
signal_name = "dbook."+str(period)

In [ ]:
train_sample = np.array(all_dates)<"2019"

In [ ]:
sum(train_sample)
## 487

In [ ]:
test_sample = np.array(all_dates)>"2019"
sum(test_sample)
## 576

In [ ]:
%%time
signal_name = "dbook."+str(period)
all_trade_stat = dict([])
for product in product_list:
    spread = product_info[product]["spread"]
    tranct = product_info[product]["tranct"]
    tranct_ratio = product_info[product]["tranct.ratio"]
    with dask.config.set(scheduler='processes', num_workers=CORE_NUM):
        f_par = functools.partial(get_signal_pnl, product=product, signal_name=signal_name, thre_mat=thre_mat,
                                 reverse=-1, tranct=tranct, max_spread=spread+0.1, tranct_ratio=tranct_ratio, 
                                       HEAD_PATH=HEAD_PATH, SIGNAL_PATH=SAVE_PATH,
                                  atr_filter=0.01)
        result = compute([delayed(f_par)(file) for file in all_dates])[0]
    trade_stat = get_hft_summary(result, thre_mat, n_days)
    all_trade_stat[product] = trade_stat


#Wall time: 20min 26s



In [ ]:
save(all_trade_stat, HEAD_PATH+"/"+"black_trade_stat_"+signal_name+".pkl")

In [ ]:
all_trade_stat = load(HEAD_PATH+"/"+"black_trade_stat_"+signal_name+".pkl")

- get the statistics result of training sample

In [ ]:
all_trade_stat = load(HEAD_PATH+"/"+"black_trade_stat_"+signal_name+".pkl")
date_str = [n[0:8] for n in all_dates]

format_dates = np.array([pd.to_datetime(d) for d in date_str])

- let's see how many good signals there are

In [ ]:
i = 0
for product in product_list:
    spread = product_info[product]["spread"]
    trade_stat = all_trade_stat[product]
    good_strat = trade_stat["final.result"]["avg.pnl"]>2*spread
    if sum(good_strat)>2:
        good_pnl = trade_stat["daily.pnl"].loc[:, good_strat].sum(axis=1)/sum(good_strat)
        i = i+1
        plt.figure(i, figsize=(16, 10));
        plt.title("");
        plt.xlabel("date");
        plt.ylabel("pnl");
        plt.title(product);
        plt.plot(format_dates, good_pnl.cumsum());

- we can see the result is quite good
- now we can split the samples into training and testing sets

In [ ]:
%%time
train_trade_stat = dict([])
for product in product_list:
    spread = product_info[product]["spread"]
    tranct = product_info[product]["tranct"]
    tranct_ratio = product_info[product]["tranct.ratio"]
    with dask.config.set(scheduler='processes', num_workers=CORE_NUM):
        f_par = functools.partial(get_signal_pnl, product=product, signal_name=signal_name, thre_mat=thre_mat,
                                 reverse=-1, tranct=tranct, max_spread=spread+0.1, tranct_ratio=tranct_ratio, 
                                  HEAD_PATH=HEAD_PATH, SIGNAL_PATH=SAVE_PATH,
                                  atr_filter=0.01)
        result = compute([delayed(f_par)(file) for file in np.array(all_dates)[train_sample]])[0]
    trade_stat = get_hft_summary(result, thre_mat, sum(train_sample))
    train_trade_stat[product] = trade_stat
    
## Wall time: 11min 43s

In [ ]:
save(train_trade_stat, HEAD_PATH+"/" + signal_name+"_black_train_trade_stat.pkl")

In [ ]:
i = 0
train_trade_stat = load(HEAD_PATH+"/" + signal_name +"_black_train_trade_stat.pkl")
for product in product_list:
    spread = product_info[product]["spread"]
    trade_stat = train_trade_stat[product]
    good_strat = (trade_stat["final.result"]["avg.pnl"]>2*spread) & (trade_stat["final.result"]["num"]>20)
    if sum(good_strat)>0:
        good_pnl = trade_stat["daily.pnl"].loc[:, good_strat].sum(axis=1)/sum(good_strat)
        i = i+1
        plt.figure(i, figsize=(16, 10));
        plt.title("");
        plt.xlabel("date");
        plt.ylabel("pnl");
        plt.title(product);
        plt.plot(format_dates[train_sample], good_pnl.cumsum());

In [ ]:
%%time
test_trade_stat = dict([])
for product in product_list:
    spread = product_info[product]["spread"]
    tranct = product_info[product]["tranct"]
    tranct_ratio = product_info[product]["tranct.ratio"]
    with dask.config.set(scheduler='processes', num_workers=CORE_NUM):
        f_par = functools.partial(get_signal_pnl, product=product, signal_name=signal_name, thre_mat=thre_mat,
                                 reverse=-1, tranct=tranct, max_spread=spread+0.1, tranct_ratio=tranct_ratio, 
                                  HEAD_PATH=HEAD_PATH, SIGNAL_PATH=SAVE_PATH,
                                  atr_filter=0.01)
        result = compute([delayed(f_par)(file) for file in np.array(all_dates)[test_sample]])[0]
    trade_stat = get_hft_summary(result, thre_mat, sum(test_sample))
    test_trade_stat[product] = trade_stat
    
# Wall time: 10min 27s


- at last we plot the selected pnl

- we can check its sharp ratio

In [ ]:
save(test_trade_stat, HEAD_PATH+"/" + signal_name + "_test_trade_stat.pkl")

- now let's check how it performs on the test set

In [ ]:
i = 0
test_all_pnl = np.zeros([sum(test_sample), len(product_list)])
train_all_pnl = np.zeros([sum(train_sample), len(product_list)])
for product in product_list:
    spread = product_info[product]["spread"]
    trade_stat = train_trade_stat[product]
    good_strat = (trade_stat["final.result"]["avg.pnl"]>2*spread) & (trade_stat["final.result"]["num"]>20)
    if sum(good_strat)>0:
        train_pnl = trade_stat["daily.pnl"].loc[:, good_strat].sum(axis=1)/sum(good_strat)
        train_std = np.std(train_pnl)
        train_pnl = train_pnl/train_std
        trade_stat = test_trade_stat[product]
        test_pnl = trade_stat["daily.pnl"].loc[:, good_strat].sum(axis=1)/sum(good_strat)/train_std
        print(product, "train sharpe ", sharpe(train_pnl), "test sharpe ", sharpe(test_pnl))
        test_all_pnl[:,i] = test_pnl
        train_all_pnl[:,i] = train_pnl
        i = i+1
        plt.figure(i, figsize=(16, 10));
        plt.title("");
        plt.xlabel("date");
        plt.ylabel("pnl");
        plt.title(product);
        plt.plot(format_dates[test_sample], test_pnl.cumsum());
    
# rb train sharpe  1.8288563078081799 test sharpe  -0.46754268375977814
# hc train sharpe  0.3748390081011434 test sharpe  0.7961904818127117
# j train sharpe  0.7640820968299172 test sharpe  0.5321634012899987
# jm train sharpe  1.1825140445777882 test sharpe  0.004681210760919914




In [ ]:
train_portfolio = np.array(np.mean(train_all_pnl, axis=1))
test_portfolio = np.array(np.mean(test_all_pnl, axis=1))
all_portfolio = np.append(train_portfolio, test_portfolio)
all_portfolio
plt.figure(1, figsize=(16, 10));
plt.title("");
plt.xlabel("date");
plt.ylabel("pnl");
plt.title("portfolio");
plt.plot(format_dates, all_portfolio.cumsum());
plt.plot(format_dates[test_sample], all_portfolio.cumsum()[test_sample])
    

- now we see the pnl curve

In [ ]:
print(sharpe(train_portfolio), sharpe(test_portfolio))
## 1.8134022443384377 0.2512024843890688



summary:

- i performs badly in the training set, so it doesn't have any strategies for the test sets, in which it performs well though;

- signal values differ significantly for every product, it makes parameter optimization quite difficult;

- test sets perform better than training set, but it cannot break new high for a long time recently

possible solution:
- choose strategies not based on performance, we can limit the parameters to a small range, and use risk parity to allocate weights to all of them;
- we can use Bayesian Optimization to choose parameters automatically to avoid humen intervention;
- maybe adding i can make the strategies break new high recently

we can try these solutions in the future

- at last we plot the selected pnl

- now we can combine all of these to a function

In [ ]:
## get backtest result of a signal on a product's list
from collections import OrderedDict
def get_list_signal_stat(signal_name, thre_mat_list, product_list, all_dates, atr_filter_list,
                    split_str="2018", min_pnl=2, min_num=20, reverse=1):
    CORE_NUM = int(os.environ['NUMBER_OF_PROCESSORS'])
    train_sample = np.array(all_dates)<split_str ## training samples
    test_sample = np.array(all_dates)>split_str ## test samples
    date_str = [n[0:8] for n in all_dates] ## date string
    format_dates = np.array([pd.to_datetime(d) for d in date_str]) ## date in xxxx-xx-xx format
    train_trade_stat = dict([]) ## training result statistics
    print("training")
    for product in product_list:
        print(product)
        spread = product_info[product]["spread"] ## get product's bid/ask spread
        tranct = product_info[product]["tranct"] ## get product's transaction cost
        tranct_ratio = product_info[product]["tranct.ratio"] ## True: tranct cost based on notional; False: fix cost
        thre_mat = thre_mat_list[product]
        atr_filter = atr_filter_list[product]
        with dask.config.set(scheduler='processes', num_workers=CORE_NUM):
            f_par = functools.partial(get_signal_pnl, product=product, signal_name=signal_name, thre_mat=thre_mat,
                                      reverse=reverse, tranct=tranct, max_spread=spread*1.1, tranct_ratio=tranct_ratio, 
                                      HEAD_PATH=HEAD_PATH, SIGNAL_PATH=SAVE_PATH,
                                      atr_filter=atr_filter)
            train_result = compute([delayed(f_par)(file) for file in np.array(all_dates)[train_sample]])[0]
        trade_stat = get_hft_summary(train_result, thre_mat, sum(train_sample))
        train_trade_stat[product] = trade_stat
    print("testing")
    test_trade_stat = dict([])
    for product in product_list:
        print(product)
        spread = product_info[product]["spread"]
        tranct = product_info[product]["tranct"]
        tranct_ratio = product_info[product]["tranct.ratio"]
        thre_mat = thre_mat_list[product]
        atr_filter = atr_filter_list[product]
        with dask.config.set(scheduler='processes', num_workers=CORE_NUM):
            f_par = functools.partial(get_signal_pnl, product=product, signal_name=signal_name, thre_mat=thre_mat,
                                     reverse=reverse, tranct=tranct, max_spread=spread*1.1, tranct_ratio=tranct_ratio,
                                      HEAD_PATH = HEAD_PATH, SIGNAL_PATH=SAVE_PATH,
                                      atr_filter=atr_filter)
            result = compute([delayed(f_par)(file) for file in np.array(all_dates)[test_sample]])[0]
        trade_stat = get_hft_summary(result, thre_mat, sum(test_sample))
        test_trade_stat[product] = trade_stat
    result=dict([])
    result["train_trade_stat"] = train_trade_stat ## save train stat
    result["test_trade_stat"] = test_trade_stat ## save test stat
    if reverse==-1:
        save(result, HEAD_PATH+"/"+signal_name+".result.pkl") ## reversal strategy
    else:
        save(result, HEAD_PATH+"/"+signal_name+".pos.result.pkl") ## trend strategy

In [ ]:
## get train and test result, choose good strategies, plot the portfolio and result of each product

def get_list_signal_result(signal_name, product_list, all_dates, split_str="2018", reverse=1,
                           min_pnl=2, min_num=20):
    if reverse==-1:
        result = load(HEAD_PATH+"/"+signal_name+".result.pkl") ## load reversal strat statistics
    else:
        result = load(HEAD_PATH+"/"+signal_name+".pos.result.pkl") ## load trend strat statistics
    train_trade_stat = result["train_trade_stat"]
    test_trade_stat = result["test_trade_stat"]
    train_sample = np.array(all_dates)<split_str
    test_sample = np.array(all_dates)>split_str
    date_str = [n[0:8] for n in all_dates]
    format_dates = np.array([pd.to_datetime(d) for d in date_str])
    i = 0
    test_all_pnl = np.zeros([sum(test_sample), len(product_list)])
    train_all_pnl = np.zeros([sum(train_sample), len(product_list)])
    for product in product_list:
        spread = product_info[product]["spread"]
        trade_stat = train_trade_stat[product]
        ## two criterion: minimum averge pnl (unit is spread), monimum number of trades
        good_strat = (trade_stat["final.result"]["avg.pnl"]>min_pnl*spread) & (trade_stat["final.result"]["num"]>min_num)
        if sum(good_strat)>0:
            train_pnl = trade_stat["daily.ret"].loc[:, good_strat].sum(axis=1)/sum(good_strat)
            train_std = np.std(train_pnl)
            train_pnl = train_pnl/train_std
            trade_stat = test_trade_stat[product]
            test_pnl = trade_stat["daily.ret"].loc[:, good_strat].sum(axis=1)/sum(good_strat)/train_std
            print(product, "train sharpe ", sharpe(train_pnl), "test sharpe ", sharpe(test_pnl))
            test_all_pnl[:,i] = test_pnl
            train_all_pnl[:,i] = train_pnl
            i = i+1
    if i>0:
        train_portfolio = np.array(np.mean(train_all_pnl[:,:i], axis=1))
        test_portfolio = np.array(np.mean(test_all_pnl[:,:i], axis=1))
        all_portfolio = np.append(train_portfolio, test_portfolio)
        plt.figure(1, figsize=(16, 10));
        plt.title("");
        plt.xlabel("date");
        plt.ylabel("pnl");
        plt.title("portfolio");
        plt.plot(format_dates, all_portfolio.cumsum());
        plt.plot(format_dates[test_sample], all_portfolio.cumsum()[test_sample])
        signal_stat = dict([])
        signal_stat["train.stat"] = train_trade_stat
        signal_stat["test.stat"] = test_trade_stat
        print("train sharpe: ", sharpe(train_portfolio), "test sharpe: ", sharpe(test_portfolio))

In [ ]:
atr_filter_list = dict([])
for product in product_list:
    atr_filter_list[product] = 0.01

In [ ]:
reverse_list = dict([])
for product in product_list:
    reverse_list[product] = -1

In [ ]:
thre_mat_list = dict([])
for product in product_list:
    thre_mat_list[product] = thre_mat

In [ ]:
%%time
signal_stat = get_list_signal_stat("dbook.4096", thre_mat_list, product_list, all_dates, atr_filter_list, 
                                   split_str="201901", min_pnl=2, min_num=20, reverse=-1)

# training
# rb
# hc
# i
# j
# jm
# testing
# rb
# hc
# i
# j
# jm
# Wall time: 21min 55s


- we can test other signals

In [ ]:
get_list_signal_result("dbook.4096", product_list, all_dates, split_str="201901", reverse=-1,
                    min_pnl=2, min_num=20)

# rb train sharpe  1.8486535771717925 test sharpe  -0.3974948789277921
# hc train sharpe  0.4039767558101178 test sharpe  0.7994366696222475
# j train sharpe  0.659279132254138 test sharpe  0.5887522124108456
# jm train sharpe  1.1554199161980594 test sharpe  0.25199413081708244
# train sharpe:  1.7837181373974313 test sharpe:  0.5271533750763496


In [ ]:
from collections import OrderedDict
class foctor_range_pos_period(factor_template):
    factor_name = "range.pos.period"
    
    params = OrderedDict([
        ("period", np.power(2, range(12,13)))
    ])
    
    def formula(self, data, period):
        return get_range_pos(data["wpr"], data["min."+str(period)], data["max."+str(period)], period).values

- let's see waht it looks like

In [ ]:
x4_1 = foctor_range_pos_period()
## we've changed it a little bit from week 2 by adding
## xs.new <- gsub(para.name, value, xs)    
## aa <- with(data,eval(parse(text=xs.new)))
## so be aware that the name of parameter should be complex enough to avoid error

- now we can create path

In [ ]:
for product in product_list:
    create_signal_path(x4_1, product, SAVE_PATH)

- then we calculate the signal

In [ ]:
%%time
for product in product_list:
    file_list = list(map(lambda x: DATA_PATH+product+"/"+x, os.listdir(DATA_PATH + product)))
    parLapply(CORE_NUM, file_list, build_composite_signal,signal_list=x4_1, product=product, HEAD_PATH=SAVE_PATH)
;

#Wall time: 3min 48s


- then we can see how the signal looks like

In [ ]:
i = 0
for product in product_list:
    all_signal = get_all_signal(np.array(all_dates), product, "range.pos."+str(period), period)
    i = i+1
    plt.figure(i, figsize=(16, 10))
    plt.title(product)
    plt.plot(all_signal)

- there is no extreme values

In [ ]:
from collections import OrderedDict
import itertools
open_list = np.arange(0.1, 0.3, 0.01)
thre_list = []
for cartesian in itertools.product(open_list, np.array([0.2, 0.4, 0.6, 0.8, 1.0])):
    thre_list.append((cartesian[0], -cartesian[0] * cartesian[1]))
thre_list = np.array(thre_list)
thre_mat = pd.DataFrame(data=OrderedDict([("open", thre_list[:, 0]), ("close", thre_list[:, 1])]))
np.shape(thre_mat)

In [ ]:
%%time
signal_stat = get_list_signal_stat("range.pos.4096", thre_mat_list, product_list, all_dates, atr_filter_list, 
                                   split_str="2019", min_pnl=2, min_num=20, reverse=-1)

# training
# rb
# hc
# i
# j
# jm
# testing
# rb
# hc
# i
# j
# jm
# Wall time: 1h 3min 30s



In [ ]:
signal_stat = get_list_signal_result("range.pos.4096", product_list, all_dates, split_str="2019", reverse=-1, 
                                   min_pnl=1, min_num=0)


In [ ]:
from collections import OrderedDict
class foctor_nr_period(factor_template):
    factor_name = "nr.period"
    
    params = OrderedDict([
        ("period", np.power(2, range(12,13)))
    ])
    
    def formula(self, data, period):
        return zero_divide(ewma(data["ret"], period, adjust=True), ewma(data["ret"].abs(), period, adjust=True)).values

In [ ]:
x5 = foctor_nr_period()
for product in product_list:
    create_signal_path(x5, product, SAVE_PATH)

In [ ]:
%%time
for product in product_list:
    file_list = list(map(lambda x: DATA_PATH+product+"/"+x, os.listdir(DATA_PATH + product)))
    parLapply(CORE_NUM, file_list, build_composite_signal,signal_list=x5, product=product, HEAD_PATH=SAVE_PATH)
;

# Wall time: 4min 8s



In [ ]:
i = 0
period = 4096
for product in product_list:
    all_signal = get_all_signal(np.array(all_dates), product, "nr."+str(period), period)
    i = i+1
    plt.figure(i, figsize=(16, 10))
    plt.title(product)
    plt.plot(all_signal)

In [ ]:
from collections import OrderedDict
import itertools
open_list = np.arange(0.02, 0.08, 0.003)
thre_list = []
for cartesian in itertools.product(open_list, np.array([0.2, 0.4, 0.6, 0.8, 1.0])):
    thre_list.append((cartesian[0], -cartesian[0] * cartesian[1]))
thre_list = np.array(thre_list)
thre_mat = pd.DataFrame(data=OrderedDict([("open", thre_list[:, 0]), ("close", thre_list[:, 1])]))
np.shape(thre_mat)

In [ ]:
%%time
signal_stat = get_list_signal_stat("nr.4096", thre_mat_list, product_list, all_dates, atr_filter_list, 
                                   split_str="2019", min_pnl=2, min_num=20, reverse=-1)


# training
# rb
# hc
# i
# j
# jm
# testing
# rb
# hc
# i
# j
# jm
# Wall time: 22min 6s




In [ ]:
signal_stat = get_list_signal_result("nr.4096", product_list, all_dates, 
                                     split_str="2019", reverse=-1, min_pnl=2, min_num=20)


# hc train sharpe  0.8777669135426436 test sharpe  -0.34543875753655245
# j train sharpe  0.4623680157507942 test sharpe  0.25154528600416065
# jm train sharpe  0.7775510188332385 test sharpe  -0.3227882573626682
# train sharpe:  1.0865172389085747 test sharpe:  -0.17079208387732403


In [ ]:
from collections import OrderedDict
class foctor_ma_diff_period(factor_template):
    factor_name = "ma.dif.10.period"
    
    params = OrderedDict([
        ("period", np.power(2, range(12,13)))
    ])
    
    def formula(self, data, period):
        return zero_divide(ewma(data["wpr"], round(period/10), adjust=True) - ewma(data["wpr"], period, adjust=True), data["wpr"]).values

In [ ]:
x99_1 = foctor_ma_diff_period()

In [ ]:
for product in product_list:
    create_signal_path(x99_1, product, SAVE_PATH)

In [ ]:
%%time
for product in product_list:
    file_list = list(map(lambda x: DATA_PATH+product+"/"+x, os.listdir(DATA_PATH + product)))
    parLapply(CORE_NUM, file_list, build_composite_signal,signal_list=x99_1, product=product, HEAD_PATH=SAVE_PATH)
;
# Wall time: 4min 14s


In [ ]:
i = 0
period = 4096
for product in product_list:
    all_signal = get_all_signal(np.array(all_dates), product, "ma.dif.10."+str(period), period)
    i = i+1
    plt.figure(i, figsize=(16, 10))
    plt.title(product)
    plt.plot(all_signal)

In [ ]:
from collections import OrderedDict
import itertools
open_list = np.arange(0.005, 0.015, 0.0005)
thre_list = []
for cartesian in itertools.product(open_list, np.array([0.2, 0.4, 0.6, 0.8, 1.0])):
    thre_list.append((cartesian[0], -cartesian[0] * cartesian[1]))
thre_list = np.array(thre_list)
thre_mat = pd.DataFrame(data=OrderedDict([("open", thre_list[:, 0]), ("close", thre_list[:, 1])]))
np.shape(thre_mat)

In [ ]:
%%time
signal_stat = get_list_signal_stat("ma.dif.10.4096", thre_mat_list, product_list, all_dates, atr_filter_list,
                                   split_str="2019", min_pnl=2, min_num=20, reverse=-1)

# training
# rb
# hc
# i
# j
# jm
# testing
# rb
# hc
# i
# j
# jm
# Wall time: 5min 54s

In [ ]:
signal_stat = get_list_signal_result("ma.dif.10.4096", product_list, all_dates, 
                                     split_str="2019", reverse=-1, min_pnl=0, min_num=0)

In [ ]:
from collections import OrderedDict
class foctor_price_osci_period(factor_template):
    factor_name = "price.osci.period"
    
    params = OrderedDict([
        ("period", np.power(2, range(12,13)))
    ])
    
    def formula(self, data, period):
        return zero_divide(ewma(data["wpr"], round(period/20), adjust=True)-ewma(data["wpr"], period, adjust=True), data["max."+str(period)]-data["min."+str(period)]).values

In [ ]:
x113 = foctor_price_osci_period()
for product in product_list:
    create_signal_path(x113, product, SAVE_PATH)

In [ ]:
%%time
for product in product_list:
    file_list = list(map(lambda x: DATA_PATH+product+"/"+x, os.listdir(DATA_PATH + product)))
    parLapply(CORE_NUM, file_list, build_composite_signal,signal_list=x113, product=product, HEAD_PATH=SAVE_PATH)
;
# Wall time: 4min 13s

In [ ]:
i = 0
period = 4096
for product in product_list:
    all_signal = get_all_signal(np.array(all_dates), product, "price.osci."+str(period), period)
    i = i+1
    plt.figure(i, figsize=(16, 10))
    plt.title(product)
    plt.plot(all_signal)

- before the last signal, let's do some analysis of skewness and kurtosis

In [ ]:
from collections import OrderedDict
import itertools
open_list = np.arange(0.6, 1, 0.02)
thre_list = []
for cartesian in itertools.product(open_list, np.array([0.2, 0.4, 0.6, 0.8, 1.0])):
    thre_list.append((cartesian[0], -cartesian[0] * cartesian[1]))
thre_list = np.array(thre_list)
thre_mat = pd.DataFrame(data=OrderedDict([("open", thre_list[:, 0]), ("close", thre_list[:, 1])]))
np.shape(thre_mat)

In [ ]:
%%time
signal_stat = get_list_signal_stat("price.osci.4096", thre_mat_list, product_list, all_dates, atr_filter_list,
                                   split_str="2019", min_pnl=2, min_num=20, reverse=1)

# training
# rb
# hc
# i
# j
# jm
# testing
# rb
# hc
# i
# j
# jm
# Wall time: 1h 3min 12s




In [ ]:
%%time
signal_stat = get_list_signal_result("price.osci.4096", product_list, all_dates, split_str="2019", reverse=1, min_pnl=0, min_num=10)



In [ ]:
from scipy.stats import kurtosis
from scipy.stats import skew

In [ ]:
def get_signal_moment(signal, all_dates, product_list, period):
    for product in product_list:
        all_signal = get_all_signal(np.array(all_dates), product, signal+"."+str(period), period)
        print(product, "kurtosis=", kurtosis(all_signal, fisher=False), " skew=", skew(all_signal))

In [ ]:
%%time
get_signal_moment("dbook", all_dates, product_list, 4096)
# rb kurtosis= 3.3292558260138123  skew= -0.03442115177889793
# hc kurtosis= 4.077468679818463  skew= 0.46463064043994623
# i kurtosis= 2.9673450716161205  skew= 0.13572601698943454
# j kurtosis= 2.975546288388362  skew= -0.030524940731791936
# jm kurtosis= 3.6810708556503924  skew= -0.051734226143421096
# Wall time: 1min 34s


In [ ]:
%%time
get_signal_moment("trade.imb", all_dates, product_list, 4096)
# rb kurtosis= 3.29532912636656  skew= 0.12598575428566536
# hc kurtosis= 3.5664109254717826  skew= -0.13690079095921548
# i kurtosis= 3.331417755262715  skew= 0.06796812147137732
# j kurtosis= 4.167959779845711  skew= -0.15955358268532033
# jm kurtosis= 5.185580618897432  skew= 0.18431565003364936
# Wall time: 1min 58s


In [ ]:
%%time
get_signal_moment("range.pos", all_dates, product_list, 4096)

# rb kurtosis= 4.100346674596123  skew= -1.1485255129347502
# hc kurtosis= 2.3445315012956534  skew= -0.01156560438700087
# i kurtosis= 2.5885098635774373  skew= -0.03704937909560492
# j kurtosis= 2.590880667972659  skew= 0.08745425848418585
# jm kurtosis= 2.7708793751945624  skew= -0.0012361839675433892
# Wall time: 1min 34s


In [ ]:
%%time
get_signal_moment("ma.dif.10", all_dates, product_list, 4096)
# rb kurtosis= 5.975013294229626  skew= 0.008699654033963028
# hc kurtosis= 5.644079050336477  skew= 0.17260167647262747
# i kurtosis= 5.18283571461787  skew= 0.1362201200418672
# j kurtosis= 5.448917468350876  skew= -0.405684478984888
# jm kurtosis= 5.3460571355279045  skew= -0.3114411735492757
# Wall time: 1min 20s



In [ ]:
%%time
get_signal_moment("price.osci", all_dates, product_list, 4096)
# rb kurtosis= nan  skew= nan
# hc kurtosis= 2.892972439136822  skew= -0.11546752593771438
# i kurtosis= 3.6375440640845067  skew= 0.1626665768880947
# j kurtosis= 2.855694414742262  skew= 0.14700242731239987
# jm kurtosis= 3.103178716094204  skew= 0.16391147958756044
# Wall time: 1min 33s



In [ ]:
%%time
get_signal_moment("nr", all_dates, product_list, 4096)

# rb kurtosis= 3.4026706281063013  skew= 0.10929041050476321
# hc kurtosis= 3.373957878100351  skew= 0.010846374425513798
# i kurtosis= 4.081882002522303  skew= -0.09087971440204498
# j kurtosis= 3.470562821581134  skew= -0.016530696557462338
# jm kurtosis= 4.326748044474971  skew= 0.35545705760852786
# Wall time: 1min 33s


In [ ]:
class foctor_kdj_k_period(factor_template):
    factor_name = "kdj.k.period"
    
    params = OrderedDict([
        ("period", np.power(2, range(12,13)))
    ])
    
    def formula(self, data, period):
        return ewma((zero_divide(data["wpr"]-data["min."+str(period)], data["max."+str(period)]-data["min."+str(period)])-0.5)*2, round(period/5),
                   adjust=True).values
    

In [ ]:
x11 = foctor_kdj_k_period()
for product in product_list:
    create_signal_path(x11, product, SAVE_PATH)

In [ ]:
%%time
for product in product_list:
    file_list = list(map(lambda x: DATA_PATH+product+"/"+x, os.listdir(DATA_PATH + product)))
    parLapply(CORE_NUM, file_list, build_composite_signal,signal_list=x11, product=product, HEAD_PATH=SAVE_PATH)
;

## Wall time: 4min 1s


In [ ]:
i = 0
period = 4096
for product in product_list:
    all_signal = get_all_signal(np.array(all_dates), product, "kdj.k."+str(period), period)
    i = i+1
    plt.figure(i, figsize=(16, 10))
    plt.title(product)
    plt.plot(all_signal)
    print(product, "kurtosis=", kurtosis(all_signal, fisher=False), " skew=", skew(all_signal))

# rb kurtosis= 2.1508507216023722  skew= -0.07707288699538488
# hc kurtosis= 1.846707216058764  skew= -0.10306613148077597
# i kurtosis= 1.8718173272037502  skew= -0.0498125302639752
# j kurtosis= 1.951455952734895  skew= 0.020499155386247456
# jm kurtosis= 1.9062516006135382  skew= -0.023197105994119047


- we know that standard normal distribution is 3, so 3.29 is a little bit higher but still close to normal distribution
- as we see above, range.pos.signal and dbook.signal are good signals, they have kurtosis of slightly above 3
- open.int.ret.signal and nr.signal are not good signals, and they have kurtosis above 8
- nr.vanish.signal is better than nr.signal, and it has kurtosis much lower
- so when we want to improve our signals, we can focus on how to decrease the kurtosis


- kurtosis is used to measure fatness and thiness of the shape of the density distribution
- financial has fat tails, which may lead to large kurtosis value
- so we can use vanish techniques to reduce it 

In [ ]:
from collections import OrderedDict
import itertools
open_list = np.arange(0.5, 1, 0.025)
thre_list = []
for cartesian in itertools.product(open_list, np.array([0.2, 0.4, 0.6, 0.8, 1.0])):
    thre_list.append((cartesian[0], -cartesian[0] * cartesian[1]))
thre_list = np.array(thre_list)
thre_mat = pd.DataFrame(data=OrderedDict([("open", thre_list[:, 0]), ("close", thre_list[:, 1])]))
np.shape(thre_mat)

In [ ]:
%%time
signal_stat = get_list_signal_stat("kdj.k.4096", thre_mat_list, product_list, all_dates, atr_filter_list,
                                   split_str = "2019", min_pnl=2, min_num=20,reverse=1)

# training
# rb
# hc
# i
# j
# jm
# testing
# rb
# hc
# i
# j
# jm
# Wall time: 1h 8min 1s


In [ ]:
%%time
signal_stat = get_list_signal_result("kdj.k.4096", product_list, all_dates, split_str="2019", reverse=1, min_pnl=2, min_num=20)
## Wall time: 310 ms


In [ ]:
class foctor_kdj_j_period(factor_template):
    factor_name = "kdj.j.period"
    
    params = OrderedDict([
        ("period", np.power(2, range(12,13)))
    ])
    
    def formula(self, data, period):
        return ewma(ewma((zero_divide(data["wpr"]-data["min."+str(period)], data["max."+str(period)]-data["min."+str(period)])-0.5)*2, round(period/5),
                        adjust=True), round(period/5), adjust=True).values
    


In [ ]:
x12 = foctor_kdj_j_period()
for product in product_list:
    create_signal_path(x12, product, SAVE_PATH)

In [ ]:
%%time
for product in product_list:
    file_list = list(map(lambda x: DATA_PATH+product+"/"+x, os.listdir(DATA_PATH + product)))
    parLapply(CORE_NUM, file_list, build_composite_signal,signal_list=x12, product=product, HEAD_PATH=SAVE_PATH)
;
# Wall time: 4min 18s

In [ ]:
%%time
i = 0
period = 4096
for product in product_list:
    all_signal = get_all_signal(np.array(all_dates), product, "kdj.j."+str(period), period)
    i = i+1;
    plt.figure(i, figsize=(16, 10));
    plt.title(product);
    plt.plot(all_signal);
    print(product, "kurtosis=", kurtosis(all_signal, fisher=False), " skew=", skew(all_signal));

# rb kurtosis= 2.0320613476981717  skew= -0.06074485121971566
# hc kurtosis= 1.9621333268608645  skew= -0.051279062622282955
# i kurtosis= 2.118568770639969  skew= -0.0918249282999728
# j kurtosis= 2.0742280155841635  skew= -0.08541220648515659
# jm kurtosis= 2.0242274171440786  skew= -0.041576092549169115
# Wall time: 1min 21s



In [ ]:
from collections import OrderedDict
import itertools
open_list = np.arange(0.5, 1, 0.025)
thre_list = []
for cartesian in itertools.product(open_list, np.array([0.2, 0.4, 0.6, 0.8, 1.0])):
    thre_list.append((cartesian[0], -cartesian[0] * cartesian[1]))
thre_list = np.array(thre_list)
thre_mat = pd.DataFrame(data=OrderedDict([("open", thre_list[:, 0]), ("close", thre_list[:, 1])]))
np.shape(thre_mat)

In [ ]:
%%time
signal_stat = get_list_signal_stat("kdj.j.4096", thre_mat_list, product_list, all_dates, atr_filter_list,
                                   split_str = "2019", min_pnl=2, min_num=20,reverse=1)

# training
# rb
# hc
# i
# j
# jm
# testing
# rb
# hc
# i
# j
# jm
# Wall time: 1h 6min 2s



In [ ]:
%%time
signal_stat = get_list_signal_result("kdj.j.4096", product_list, all_dates, split_str="2019", reverse=1, min_pnl=2, min_num=20)


In [ ]:
%%time
signal_stat = get_list_signal_stat("kdj.j.4096", thre_mat_list, product_list, all_dates, atr_filter_list,
                                   split_str = "2019", min_pnl=2, min_num=20,reverse=-1)


# training
# rb
# hc
# i
# j
# jm
# testing
# rb
# hc
# i
# j
# jm
# Wall time: 1h 6min 45s


In [ ]:
%%time
signal_stat = get_list_signal_result("kdj.j.4096", product_list, all_dates, split_str="2019", reverse=1, min_pnl=2, min_num=20)